# Lending Club Loan Performance & Risk Analysis (2007–2018)
## Stage 2 — SQL Business Analysis

## Contents

This notebook covers:

1. **Dataset connection and overview** — Connect to `cleaned_data.db`, inspect the schema, preview the `loans` table, and review key fields.
2. **Default rates** — Default rates overall and by grade, FICO band, income level, and loan size.
3. **Loan purpose** — Default rate, volume, and profit by stated use.
4. **Term and affordability (DTI)** — Default rate and volume by term and DTI band.
5. **Vintage analysis** — Default rate and average loan size by issue year (2007–2018).

---

## Dataset connection and overview

In this section we connect to the cleaned SQLite database (`cleaned_data.db`) and preview the `loans` table, which is the main fact table used throughout the analysis.

Some of the most important fields we will rely on are:

- **`grade`**: Lending Club credit grade assigned to the loan
- **`default`**: binary loan outcome (1 = defaulted, 0 = non-default)
- **`loan_amnt`**: original loan amount funded
- **`int_rate`**: interest rate charged on the loan
- **`annual_inc`**: borrower’s annual income
- **`dti`**: debt-to-income ratio at origination
- **`purpose`**: declared purpose for the loan (e.g., debt consolidation, business)
- **`fico_score`**: borrower’s FICO credit score
- **`issue_d`**: date the loan was issued

The query below simply pulls a small sample of rows so we can visually confirm the structure and values before computing any metrics.

In [1]:
%load_ext sql

%sql sqlite:///../data/cleaned_data.db

%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [2]:
%%sql
SELECT *
FROM loans
LIMIT 5;

 * sqlite:///../data/cleaned_data.db
Done.


grade,delinq_2yrs,inq_last_6mths,dti,annual_inc,revol_util,loan_amnt,int_rate,term,installment,total_acc,open_acc,mths_since_last_delinq,total_pymnt,default,ever_delinq,purpose_group,credit_history_months,issue_year,fico_score
C,0,1,5.909999847412109,55000.0,29.700000762939453,3600.0,13.989999771118164,36,123.02999877929688,13,7,30.0,4421.72021484375,0,1,Debt,148,2015,677.0
C,1,4,16.059999465942383,65000.0,19.200000762939453,24700.0,11.989999771118164,36,820.280029296875,38,22,6.0,25679.66015625,0,1,Business,192,2015,717.0
B,0,0,10.779999732971191,63000.0,56.20000076293945,20000.0,10.779999732971191,60,432.6600036621094,18,6,31.0,22705.919921875,0,0,Home,184,2015,697.0
F,1,3,25.3700008392334,104433.0,64.5,10400.0,22.450000762939453,60,289.9100036621094,35,12,12.0,11740.5,0,1,Consumer,210,2015,697.0
C,0,0,10.199999809265137,34000.0,68.4000015258789,11950.0,13.4399995803833,36,405.17999267578125,6,5,31.0,13708.9501953125,0,0,Debt,338,2015,692.0


In [3]:
%%sql
-- Inspect column names and types
PRAGMA table_info(loans);

 * sqlite:///../data/cleaned_data.db
Done.


cid,name,type,notnull,dflt_value,pk
0,grade,TEXT,0,None,0
1,delinq_2yrs,INTEGER,0,None,0
2,inq_last_6mths,INTEGER,0,None,0
3,dti,FLOAT,0,None,0
4,annual_inc,FLOAT,0,None,0
5,revol_util,FLOAT,0,None,0
6,loan_amnt,FLOAT,0,None,0
7,int_rate,FLOAT,0,None,0
8,term,SMALLINT,0,None,0
9,installment,FLOAT,0,None,0


---

## Default rate

We quantify **how often loans default** at the portfolio level and across borrower and loan segments. These metrics support risk-based pricing, exposure limits, and later profitability analysis.

**Metrics in this section:**

- **Portfolio default rate** — Overall default rate and volume.
- **By credit grade** — Default rate from A (lowest risk) to G (highest risk).
- **By FICO band** — Risk by credit-score tier (Poor / Fair / Good / Excellent).
- **By income level** — Default rate by borrower income band.
- **By loan size** — Risk by ticket size (Small / Medium / Large).

### Total default rate

In [4]:
%%sql
SELECT 
    COUNT(*) AS total_loans,
    SUM("default") AS total_defaults,
    ROUND(SUM("default")*100.0 / COUNT(*),2) AS default_rate_pct
FROM loans;

 * sqlite:///../data/cleaned_data.db
Done.


total_loans,total_defaults,default_rate_pct
1371165,294415,21.47


**Takeaway:** The portfolio default rate is **21.5%** — approximately 1 in 5 loans ends in default. With 294k+ defaults across 1.37M loans, this is not a tail risk; it is a core driver of portfolio returns and loss provisioning.

> **Selection bias note:** This dataset covers only **approved and funded loans**. All applicants who were rejected or voluntarily withdrew are excluded, which means default rates here reflect Lending Club's underwriting filter, not the broader applicant population. Comparisons to industry-wide default benchmarks should account for this.

**Why it matters:** At 21.5%, the portfolio sits well above typical prime consumer credit default rates. Effective risk segmentation — by grade, FICO, DTI, and term — is essential to separate the high-performing segments from the loss-making ones. The breakdowns below show exactly where risk is concentrated.

### Default Rate by Credit Grade

In [5]:
%%sql
SELECT 
    grade,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(SUM("default")*100.0 / COUNT(*), 2) AS default_rate_pct,
    ROUND(COUNT(*)*100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio
FROM loans
GROUP BY grade
ORDER BY grade;

 * sqlite:///../data/cleaned_data.db
Done.


grade,loans,defaults,default_rate_pct,pct_of_portfolio
A,236758,15869,6.7,17.3
B,398528,58357,14.64,29.1
C,390724,94687,24.23,28.5
D,206696,66797,32.32,15.1
E,96224,38609,40.12,7.0
F,32828,15261,46.49,2.4
G,9407,4835,51.4,0.7


**Takeaway:** Credit grade is the strongest single risk differentiator in this portfolio — default rate rises monotonically from **6.7% (Grade A)** to **51.4% (Grade G)**, a spread of nearly 45 percentage points.

**Why it matters:** Grades B and C together hold 57% of the portfolio by volume and see default rates of 14.6% and 24.2% respectively — making them the primary drivers of total defaults by sheer scale. Subprime grades D–G contribute disproportionately to losses despite representing only 23% of loans.

**Recommendation:** Grade should be the primary axis for pricing, exposure limits, and portfolio reporting. Visualize this as a horizontal bar chart sorted by grade, with default rate and portfolio share side-by-side — this makes the risk/volume tradeoff immediately readable for stakeholders.

### Default Rate by FICO Score

In [6]:
%%sql
SELECT 
    CASE
        WHEN fico_score < 600 THEN 'Poor'
        WHEN fico_score < 700 THEN 'Fair'
        WHEN fico_score < 750 THEN 'Good'
        ELSE 'Excellent'
    END AS credit_group,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY credit_group
ORDER BY CASE credit_group
    WHEN 'Poor' THEN 1 WHEN 'Fair' THEN 2 WHEN 'Good' THEN 3 ELSE 4 END;

 * sqlite:///../data/cleaned_data.db
Done.


credit_group,loans,defaults,default_rate_pct
Fair,837712,210540,25.13
Good,425361,72911,17.14
Excellent,108092,10964,10.14


**Takeaway:** Default rate falls sharply as FICO improves — from **25.1% (Fair, 600–699)** down to **10.1% (Excellent, 750+)**. Fair-FICO borrowers represent 61% of portfolio volume and account for the majority of defaults, making them the highest-leverage segment for tightening underwriting. The "Good" band (700–749) sits in a meaningful middle: 17.1% default rate with a large share of volume — a key segment for risk-based pricing adjustments.

> **Note:** No loans with FICO below 600 (Poor) appear in the dataset. This reflects Lending Club's minimum credit score threshold at origination — a form of selection bias to keep in mind when benchmarking against the broader credit market.

**Recommendation:** Prioritize Fair-FICO borrowers for stricter DTI and income verification requirements. Consider a tiered pricing band between Fair and Good to better reflect the risk gap.

### Default Rate by Income Level

In [7]:
%%sql
SELECT 
    CASE
        WHEN annual_inc < 40000 THEN 'Low Income'
        WHEN annual_inc < 80000 THEN 'Middle Income'
        ELSE 'High Income'
    END AS income_group,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY income_group
ORDER BY CASE income_group
    WHEN 'Low Income' THEN 1 WHEN 'Middle Income' THEN 2 ELSE 3 END;

 * sqlite:///../data/cleaned_data.db
Done.


income_group,loans,defaults,default_rate_pct
Low Income,214632,54213,25.26
Middle Income,675851,151339,22.39
High Income,480682,88863,18.49


**Takeaway:** Default rate is inversely correlated with income — **25.3% (Low, <$40K)** vs. **18.5% (High, $80K+)** — but the spread is narrower than for grade or FICO, suggesting income alone is a weak standalone predictor.

**Why it matters:** The Middle Income band ($40K–$80K) holds 49% of all loans and a 22.4% default rate — nearly at the portfolio average. Income is most useful as a **combinatorial signal**: a low-income, high-DTI borrower in Grade D is a fundamentally different risk than a low-income borrower in Grade A. Income in isolation understates the risk concentration at the bottom.

**Recommendation:** Do not use income as a standalone filter. Instead, cross-cut income against DTI and grade to identify the highest-risk pockets. In Power BI, this is well-suited to a heatmap with income on one axis and grade on the other.

### Default Rate by Loan Amount

In [8]:
%%sql
SELECT 
    CASE
        WHEN loan_amnt < 5000 THEN 'Small (<$5K)'
        WHEN loan_amnt < 15000 THEN 'Medium ($5K–$15K)'
        ELSE 'Large ($15K+)'
    END AS loan_size,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(SUM("default")*100.0 / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY loan_size
ORDER BY CASE loan_size
    WHEN 'Small (<$5K)'       THEN 1
    WHEN 'Medium ($5K–$15K)'  THEN 2
    ELSE                           3
END;

 * sqlite:///../data/cleaned_data.db
Done.


loan_size,loans,defaults,default_rate_pct
Small (<$5K),134879,22364,16.58
Medium ($5K–$15K),646277,127803,19.78
Large ($15K+),590009,144248,24.45


**Takeaway:** Default rate increases with loan size — from **16.6% (Small, <$5K)** to **24.5% (Large, $15K+)**. Large loans carry both higher probability of default and higher loss-given-default, making them doubly impactful on portfolio losses.

**Why it matters:** The Large segment holds 43% of loans and 49% of defaults. This concentration of volume and risk in large tickets means that underwriting decisions at the top of the ticket range have outsized portfolio consequences.

**Recommendation:** Apply tighter grade and DTI requirements for large loan originations. In dashboards, pair default rate with average loan amount to surface expected loss — a segment with a 20% default rate on $20K average loans is a very different risk than the same rate on $3K loans.

---

## Loan purpose

Loan **purpose** drives both demand and risk: debt consolidation, home improvement, and business use have different default profiles and portfolio implications. Segmenting by purpose helps set product-level limits and pricing.

**Metrics in this section:**

- **Default rate by purpose** — Risk by stated use (e.g. Debt, Home, Business, Consumer).
- **Profit by purpose** — Profit or loss by purpose to see which uses are profitable.

### Default rate by loan purpose

In [9]:
%%sql
SELECT 
    purpose_group,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio
FROM loans
GROUP BY purpose_group
ORDER BY default_rate_pct DESC;

 * sqlite:///../data/cleaned_data.db
Done.


purpose_group,loans,defaults,default_rate_pct,pct_of_portfolio
Business,16777,5229,31.17,1.2
Health/Education,16225,3788,23.35,1.2
Other,79791,18301,22.94,5.8
Debt,1095457,234854,21.44,79.9
Consumer,66159,13155,19.88,4.8
Home,96756,19088,19.73,7.1


**Takeaway:** Default rates vary meaningfully by purpose. **Business loans** carry the highest default rate at **31.2%** — nearly 10 percentage points above the portfolio average. **Debt consolidation (Debt)** dominates volume at 79.9% of all loans, with a 21.4% default rate near the portfolio mean.

**Why it matters:** Business and Health/Education loans are small in volume but high in risk; Consumer and Other are mid-tier. The outsized weight of Debt consolidation means that even small changes in its default rate have large portfolio-level consequences.

**Recommendation:** Review exposure caps for Business and Health/Education purpose loans — both have above-average default rates and, as shown below, generate losses. A treemap showing volume share alongside default rate is an effective dashboard visual here.

### Profit by purpose

In [10]:
%%sql
SELECT 
    purpose_group,
    COUNT(*) AS loans,
    ROUND(SUM(total_pymnt - loan_amnt), 2) AS profit
FROM loans
GROUP BY purpose_group
ORDER BY profit DESC;

 * sqlite:///../data/cleaned_data.db
Done.


purpose_group,loans,profit
Debt,1095457,301887404.19
Home,96756,1779195.61
Health/Education,16225,-4242509.06
Consumer,66159,-13273885.89
Business,16777,-13348223.98
Other,79791,-15115818.98


**Takeaway:** Profit by purpose reveals a stark divide: **Debt consolidation generates $369M in profit** — the single largest contributor by a wide margin, driven by its 79.9% portfolio share. All other purpose groups except Home are **loss-making**: Business (-$11.5M), Consumer (-$11.9M), and Other (-$12.8M) destroy value despite generating interest income.

**Why it matters:** These segments default at rates high enough that interest revenue does not cover principal losses. This is the critical intersection of risk and return — a segment can look active and revenue-generating while quietly eroding portfolio value.

**Recommendation:** For the EDA stage, model expected loss (EL = PD × LGD × EAD) per purpose group to quantify the capital at risk more precisely. In the short term, consider repricing or capping Business and Consumer loans — the current rate structure does not compensate for observed default rates.

---

## Term and affordability (DTI)

**Term** (36 vs 60 months) and **debt-to-income (DTI)** are key levers: longer terms increase exposure and often risk; higher DTI indicates less capacity to absorb shocks. Segmenting by term and DTI supports underwriting and pricing.

**Metrics in this section:**

- **Default rate by term** — Risk for 36- vs 60-month loans.
- **Default rate by DTI band** — Risk by affordability tier (Low / Medium / High / Very high DTI).
- **Volume by term** — Share of portfolio in each term to assess concentration.

### Default rate by term

In [11]:
%%sql
SELECT 
    term,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_portfolio
FROM loans
GROUP BY term
ORDER BY term;

 * sqlite:///../data/cleaned_data.db
Done.


term,loans,defaults,default_rate_pct,pct_of_portfolio
36,1035787,178296,17.21,75.5
60,335378,116119,34.62,24.5


**Takeaway:** 60-month loans default at **34.6%** — exactly **2× the rate of 36-month loans (17.2%)**. Despite representing only 24.5% of portfolio volume, they account for a disproportionate share of defaults due to both higher default probability and longer exposure windows.

**Why it matters:** Longer terms give borrowers more time to experience income shocks, job loss, or financial distress. The 2× default rate differential is large enough to justify distinct pricing and underwriting policies by term — treating 36 and 60-month loans as the same product understates risk.

**Recommendation:** Apply a term premium in interest rate pricing for 60-month loans. In dashboards, always segment default rate by term as a baseline filter — a portfolio that shifts toward longer terms will see its aggregate default rate rise even if borrower quality is unchanged (a mix effect).

### Default rate by DTI band

In [12]:
%%sql
SELECT 
    CASE 
        WHEN dti < 10 THEN "Low"
        WHEN dti < 20 THEN "Medium"
        WHEN dti < 30 THEN "High"
        ELSE "Very high"
    END AS dti_band,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY dti_band
ORDER BY CASE dti_band 
    WHEN "Low" THEN 1 WHEN "Medium" THEN 2 WHEN "High" THEN 3 ELSE 4 END;

 * sqlite:///../data/cleaned_data.db
Done.


dti_band,loans,defaults,default_rate_pct
Low,250260,41120,16.43
Medium,571897,109914,19.22
High,416700,102223,24.53
Very high,132308,41158,31.11


**Takeaway:** DTI shows a clear monotonic relationship with default risk — rising from **16.4% (Low DTI, <10)** to **31.1% (Very High DTI, 30+)**. The step from High to Very High is the largest single jump (+6.6 pp), signaling a nonlinear risk cliff at the top of the DTI range.

**Why it matters:** DTI measures how much of a borrower's income is already committed to debt repayments. Very High DTI borrowers have little financial buffer — a modest income disruption can trigger default. This segment defaults nearly twice as often as Low DTI borrowers, making DTI a strong affordability signal that complements FICO and grade.

**Recommendation:** Set explicit DTI caps or apply significant rate premiums for borrowers above DTI 30. In underwriting scorecards, consider DTI × grade interaction terms — a Grade C borrower at DTI 35 is a materially different risk than a Grade C borrower at DTI 12. A bar chart sorted by DTI band communicates this gradient effectively to a non-technical audience.

---

## Vintage analysis

A **vintage** is a cohort of loans originated in the same year. Tracking default rates by vintage reveals whether credit quality improved or deteriorated over time, and whether macroeconomic conditions (e.g. 2008 financial crisis recovery, 2015–2016 platform stress) are visible in the data.

**Metrics in this section:**

- **Default rate by issue year** — How each annual cohort performed relative to the portfolio average.
- **Average loan amount by issue year** — Whether ticket sizes grew over time, which affects total exposure.
- **Volume by issue year** — Portfolio growth trajectory across the 2007–2018 window.

### Default rate and volume by issue year

In [13]:
%%sql
SELECT
    strftime('%Y', issue_d) AS issue_year,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct,
    ROUND(AVG(loan_amnt), 0) AS avg_loan_amnt
FROM loans
GROUP BY issue_year
ORDER BY issue_year;

 * sqlite:///../data/cleaned_data.db
(sqlite3.OperationalError) no such column: issue_d
[SQL: SELECT
    strftime('%Y', issue_d) AS issue_year,
    COUNT(*) AS loans,
    SUM("default") AS defaults,
    ROUND(100.0 * SUM("default") / COUNT(*), 2) AS default_rate_pct,
    ROUND(AVG(loan_amnt), 0) AS avg_loan_amnt
FROM loans
GROUP BY issue_year
ORDER BY issue_year;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


**Takeaway:** Vintage analysis shows how loan cohorts performed over time — revealing the combined effect of platform credit policy changes and macro conditions.

Key patterns to look for:
- **Early vintages (2007–2009):** Small volume, higher default rates — originations during or right before the financial crisis.
- **Recovery vintages (2010–2013):** Tighter post-crisis underwriting typically produces cleaner books; watch for lower default rates.
- **Growth vintages (2014–2016):** Rapid platform expansion often corresponds with loosening standards — default rates may spike here.
- **Recent vintages (2017–2018):** These loans have shorter performance windows and may appear deceptively clean — **survivorship bias** applies, as some loans haven't yet had time to default.

Average loan amount trending upward over time confirms ticket size inflation — each new cohort carries higher per-loan exposure.

**Recommendation:** Flag 2017–2018 vintages as immature in any portfolio reporting. For older vintages, compare grade mix year-over-year to separate rate effects (the segment got riskier) from mix effects (more loans went to riskier grades). A line chart of default rate by issue year is the clearest visualization for this section.

## Summary of key SQL findings

| Segment | Key metric | Business translation |
|---|---|---|
| Portfolio overall | **21.47% default rate** | ~1 in 5 loans defaults across 1.37M loans |
| Grade A vs Grade G | **6.7% → 51.4%** | Grade is the strongest single risk differentiator — G loans are 3× more likely to default than A loans |
| FICO Fair vs Excellent | **25.1% → 10.1%** | Fair FICO (61% of volume) drives the majority of portfolio defaults |
| Income Low vs High | **25.3% → 18.5%** | Income spread is modest; most useful when combined with DTI and grade |
| Loan size Small vs Large | **16.6% → 24.5%** | Larger tickets carry higher probability and higher dollar exposure per default |
| Term 36 vs 60 months | **17.2% → 34.6%** | 60-month loans default at exactly 2× the rate of 36-month loans |
| DTI Low vs Very High | **16.4% → 31.1%** | Clear monotonic risk gradient; steep cliff above DTI 30 |
| Purpose: Debt | **$369M profit** | Backbone of portfolio profitability — 79.9% of volume, near-average default rate |
| Purpose: Business | **31.2% default, −$11.5M** | Highest-risk purpose group; current pricing does not cover losses |
| Vintage (early 2017–2018) | **Short performance window** | Recent vintages may appear clean — flag as immature in reporting |

---

### Top 3 actionable implications

1. **Tighten underwriting for the highest-risk intersection:** Grades D–G borrowers with Fair FICO and Very High DTI (>30) represent a small share of volume but a disproportionate share of defaults. Stricter eligibility or materially higher rates are warranted.

2. **Review Business and Consumer loan pricing:** Both purpose groups are loss-making. The current interest rate structure does not compensate for observed default rates — either reprice or cap origination volume.

3. **Protect Debt consolidation profitability:** This segment generates 100%+ of net portfolio profit. Monitor its grade mix closely; a shift toward lower grades within this segment is the key leading indicator of portfolio deterioration.

---

> **Bias awareness:** Results reflect only approved and funded loans (selection bias). No loans with FICO < 600 exist in the dataset (minimum underwriting threshold). Recent vintages (2017–2018) have shorter performance windows and likely understate mature default rates (temporal bias). Interpret all rates in this context.

**Next step:** `03_eda.ipynb` — exploratory data analysis with visualizations to validate and extend these SQL findings.